In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rasterio.features
import geopandas as gpd
import math
from shapely.geometry import shape
from datetime import datetime
from pytz import timezone, utc
import matplotlib.patches as patches

# --- Parameters ---
original_root = '/Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR'

# Use the folder that actually contains your mask files
mask_root = '/Volumes/External/TJ/040_segoutput'
# mask_root = '/Volumes/External/TJ/030_autoSARoutput/ContrastRatio'

output_plot_dir = '/Volumes/External/TJ/050_viz/autoSAR_output'
output_shapefile_dir = '/Volumes/External/TJ/015_shapefiles/autoSAR_output'

met_csv_path = '/Volumes/External/TJ_estuary/analysis/TJRTLMET.csv'
threshold = 0

# Make sure this is the full absolute path
point_shapefile_path = '/Volumes/External/TJ/015_shapefiles/outflow_PB_TJ/Outflow.shp'

pacific = timezone('US/Pacific')

# --- Load meteorological data ---
met_df = pd.read_csv(met_csv_path)
met_df.columns = met_df.columns.str.strip()

met_df['DateTimeStamp'] = pd.to_datetime(
    met_df['DateTimeStamp'],
    format='%m/%d/%y %H:%M',
    errors='coerce'
)

met_df['DateTimeStamp'] = met_df['DateTimeStamp'].dt.tz_localize(
    pacific,
    ambiguous='NaT',
    nonexistent='NaT'
)

met_df = met_df.dropna(subset=['DateTimeStamp']).sort_values('DateTimeStamp')

# --- Load point data once ---
points_gdf_orig = gpd.read_file(point_shapefile_path)


def extract_datetime_from_filename(filename):
    """
    Extract UTC datetime from Sentinel-1 filename.
    Example:
    S1A_IW_GRDH_1SDV_20220101T015007_...
    """
    parts = filename.split('_')
    for part in parts:
        if part.startswith('20') and 'T' in part:
            try:
                return utc.localize(datetime.strptime(part, '%Y%m%dT%H%M%S'))
            except ValueError:
                continue
    return None


def get_nearest_met_data(local_time):
    """
    Find the closest met record to the given local time.
    """
    diffs = (met_df['DateTimeStamp'] - local_time).abs()
    idx = diffs.idxmin()
    row = met_df.loc[idx]
    return row['WSpd'], row['Wdir'], row['DateTimeStamp']


def match_original_path(mask_filename):
    """
    Convert mask filename like:

    S1A_..._pre_reordered_JPL1.3_VVDR_cumulative_mask.tif

    to original filename like:

    S1A_..._pre_reordered.tif

    This works for JPL0.4, JPL1.3, JPL2.0, etc.
    """
    original_filename = re.sub(
        r'_JPL[\d.]+_VVDR_cumulative_mask\.tif$',
        '.tif',
        mask_filename
    )

    return os.path.join(original_root, original_filename)


# --- Make output directories ---
os.makedirs(output_plot_dir, exist_ok=True)
os.makedirs(output_shapefile_dir, exist_ok=True)


# --- Main processing loop ---
for root, _, files in os.walk(mask_root):
    for file in files:
        if file.startswith('.') or not file.endswith('_VVDR_cumulative_mask.tif'):
            continue

        mask_path = os.path.join(root, file)
        original_path = match_original_path(file)

        if not os.path.exists(original_path):
            print(f'[!] Original image not found for mask: {file}')
            print(f'    Expected original path: {original_path}')
            continue

        print(f'\nProcessing: {file}')

        # --- Read mask ---
        with rasterio.open(mask_path) as src_mask:
            mask_arr = src_mask.read(1)
            transform = src_mask.transform
            crs = src_mask.crs
            nodata = src_mask.nodata

        valid_mask = mask_arr > threshold

        if nodata is not None:
            valid_mask &= mask_arr != nodata

        # --- Vectorize mask ---
        shapes = []

        for geom, val in rasterio.features.shapes(
            mask_arr.astype('int16'),
            mask=valid_mask,
            transform=transform
        ):
            shapes.append({
                'geometry': shape(geom),
                'value': int(val)
            })

        if shapes:
            gdf = gpd.GeoDataFrame(shapes, crs=crs)

            shp_name = re.sub(
                r'_JPL[\d.]+_VVDR_cumulative_mask\.tif$',
                '_mask.shp',
                file
            )

            out_shp = os.path.join(output_shapefile_dir, shp_name)
            gdf.to_file(out_shp, driver='ESRI Shapefile')

            print(f'  wrote shapefile: {out_shp}')
        else:
            print(f'  no valid mask pixels for: {file}, skipping shapefile.')

        # --- Get image time and met data ---
        image_time_utc = extract_datetime_from_filename(file)

        if image_time_utc is None:
            print(f'[!] Could not extract datetime from filename: {file}')
            continue

        image_time_local = image_time_utc.astimezone(pacific)
        wspd, wdir, met_time = get_nearest_met_data(image_time_local)

        # --- Read original image ---
        with rasterio.open(original_path) as src_orig:
            original = src_orig.read(1).astype(float)
            bounds = src_orig.bounds
            extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
            orig_crs = src_orig.crs

        # --- Stretch original image ---
        vmin, vmax = np.nanpercentile(original, (2, 95))

        if vmax == vmin:
            print(f'[!] Image has no contrast range: {original_path}')
            continue

        clipped = np.clip(original, vmin, vmax)
        normed = (clipped - vmin) / (vmax - vmin)

        # --- Build RGBA overlay ---
        overlay = np.zeros((mask_arr.shape[0], mask_arr.shape[1], 4))
        overlay[valid_mask] = [1.0, 1.0, 0.8, 1.0]

        # --- Reproject points to match image CRS ---
        points_gdf = points_gdf_orig.to_crs(orig_crs)

        # --- Plot ---
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(9, 12),
            constrained_layout=True
        )

        # Left panel
        axes[0].imshow(normed, cmap='gray', extent=extent)
        points_gdf.plot(
            ax=axes[0],
            marker='o',
            color='red',
            markersize=5,
            label='Outflow'
        )
        axes[0].set_title('Original 2–95% Stretch', fontsize=12)
        axes[0].axis('off')

        # Right panel
        axes[1].imshow(normed, cmap='gray', extent=extent)
        axes[1].imshow(overlay, extent=extent)
        points_gdf.plot(
            ax=axes[1],
            marker='o',
            color='red',
            markersize=5
        )
        axes[1].set_title('Original with Plume Mask', fontsize=12)
        axes[1].axis('off')

        # --- Annotation ---
        wind_text = (
            f'Wind: {wspd:.1f} m/s @ {wdir:.0f}°\n'
            f'Image Local Time: {image_time_local.strftime("%Y-%m-%d %H:%M")}\n'
            f'Nearest Met Time: {met_time.strftime("%Y-%m-%d %H:%M")}'
        )

        fig.text(
            0.5,
            0.15,
            wind_text,
            ha='center',
            fontsize=13,
            fontweight='bold'
        )

        # --- Wind vector ---
        wind_angle_deg = (270 - wdir) % 360
        angle_rad = math.radians(wind_angle_deg)

        height = bounds.top - bounds.bottom
        width = bounds.right - bounds.left

        base_frac = 0.07
        scale = np.clip(wspd / 10, 0.5, 1.5)
        arrow_length = height * base_frac * scale

        dx = arrow_length * math.cos(angle_rad)
        dy = arrow_length * math.sin(angle_rad)

        x0 = bounds.right - 0.1 * width
        y0 = bounds.bottom + 0.05 * height

        axes[1].add_patch(
            patches.FancyArrow(
                x0,
                y0,
                dx,
                dy,
                width=arrow_length * 0.05,
                head_width=arrow_length * 0.15,
                head_length=arrow_length * 0.15,
                length_includes_head=True,
                transform=axes[1].transData,
                color='lime',
                alpha=0.9
            )
        )

        # --- Save figure ---
        flat_name = re.sub(
            r'_JPL[\d.]+_VVDR_cumulative_mask\.tif$',
            '_overlay.png',
            file
        )

        out_png = os.path.join(output_plot_dir, flat_name)

        plt.savefig(out_png, dpi=150, bbox_inches='tight')
        plt.close()

        print(f'  saved plot: {out_png}')

print('Done!')


[!] Original image not found for mask: subset_1_of_subset_0_of_S1C_IW_GRDH_1SDV_20251013T134353_20251013T134418_004544_008FD8_3212_Cal_ML_EC_msk_msk_clip_JPLv0.0.1_VVDR_cumulative_mask.tif
    Expected original path: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/subset_1_of_subset_0_of_S1C_IW_GRDH_1SDV_20251013T134353_20251013T134418_004544_008FD8_3212_Cal_ML_EC_msk_msk_clip_JPLv0.0.1_VVDR_cumulative_mask.tif

Processing: S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_JPL1.3_VVDR_cumulative_mask.tif
  wrote shapefile: /Volumes/External/TJ/015_shapefiles/autoSAR_output/S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_mask.shp
  saved plot: /Volumes/External/TJ/050_viz/autoSAR_output/S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_overlay.png

Processing: S1A_IW_GRDH_1SDV_20220101T135309_20220101T135334_041268_04E7A6_B4CB_pre_reordered_JPL1.3_VVDR_cumulative_mask